# Part 2 — A multi-agent client 

Part 1 (`spectra-mcp-server`) gave us a server full of cosmology tools.
This notebook builds the **client**: a small LangGraph multi-agent system that
takes a one-sentence science question, plans, calls the server's tools, and
reproduces the ground-truth figure.

```
 science question
        │
        ▼
   ┌────────┐  plan   ┌──────────┐
   │  lead  │ ──────► │  worker  │──┐
   │        │ ◄────── │          │◄─┘ one step per visit,
   └────────┘  done   └──────────┘    calls MCP tools
        │
        ▼                  ▲ ▲
     report       MCP (HTTP or stdio)
                           │ │
                 spectra-mcp-server   ← CLASS, data, matplotlib live HERE
```

Two subagent (LLM roles) — think supervisor and grad student. The **lead** is visited twice:
first it turns the task into a step-by-step plan, and after all the work is
done it writes the report. The **worker** executes one step per visit by
calling MCP tools. The whole client is ~300 lines in `agents/` — read it
alongside this notebook.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `import agents` works

from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")  # reads GOOGLE_API_KEY

OUTPUT_DIR = str((Path.cwd().parent / "agent-output").resolve())

## 1. Connect to the server (streamable HTTP)

Start the server **in a terminal** first, from the `spectra-mcp-server` repo:

```bash
python -m mcp_server --transport streamable-http --port 8000
```

The server is now a process *you* own, visible in your terminal — that's the
client-server split. We connect to it as a URL.

> Note: the server CLI says `streamable-http` (hyphen), the
> LangChain adapter config says `streamable_http` (underscore).

In [2]:
from agents import load_tools

HTTP_CONFIG = {
    "spectra": {
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8000/mcp",
    }
}

tools = await load_tools(HTTP_CONFIG)
for t in tools:
    print(f"{t.name:25s} {t.description.strip().splitlines()[0]}")

get_eboss_data            Return the bundled eBOSS DR14 Lyman-alpha forest power spectrum data.
list_cosmology_models     List the cosmological models available to compute_power_spectrum.
compute_power_spectrum    Compute a linear matter power spectrum P(k) with the CLASS Boltzmann code.
plot_power_spectra        Plot model power spectra against the eBOSS DR14 Ly-a forest data.


Those four tools — and their descriptions, argument types, and constraints —
came from the server's `__all__` + type hints. Nothing about
cosmology is written into this client.

**Check**: call a tool directly, no LLM involved yet.

In [3]:
tools_by_name = {t.name: t for t in tools}

raw = await tools_by_name["get_eboss_data"].ainvoke({})
print(str(raw)[:400], "...")

[{'type': 'text', 'text': '{\n  "status": "success",\n  "files": [\n    "/Users/nesar/Projects/Tutorials/spectra-mcp-server/data/DR14_pm3d_19kbins.txt"\n  ],\n  "message": "Loaded 19 eBOSS DR14 Ly-a P(k) bins.",\n  "metadata": {\n    "label": "eBOSS DR14 Ly-a forest",\n    "k_h_per_Mpc": [\n      0.3084,\n      0.4992,\n      0.6899,\n      0.8807,\n      1.0715,\n      1.2622,\n      1.453,\n     ...


## 2. The LLM

Gemini (via the `gemini-flash-latest` alias) through Google's **OpenAI-compatible** endpoint (free key from
[AI Studio](https://aistudio.google.com/apikey)). Because the client only speaks
the OpenAI protocol, swapping backends is one argument — notebook 03 runs the
whole system on Groq's free open-weight `gpt-oss-120b` with `make_llm("groq")`,
and any other compatible endpoint is one more entry in `agents/llm.py`.

You may ask the LLM a direct question using `llm.invoke()'. This is somewhat similar to using Claude/ChatGPT GUI. 

In [4]:
from agents import make_llm

llm = make_llm()
llm.invoke("One sentence: why do massive neutrinos suppress small-scale structure?").content

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 12.932929876s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'}, '

## 3. The graph

Two nodes share one small state dict (`agents/state.py`):

| node | LLM? | job |
|---|---|---|
| `lead` | yes | 1st visit: turn the task into a JSON list of 2–5 steps. Last visit: write the report |
| `worker` | yes | execute ONE step by calling MCP tools (≤6 tool rounds) |

The edges are deterministic Python — `route_from_lead` / `route_from_worker`
in `agents/graph.py` loop the worker until the plan is exhausted, then send
control back to the lead. Not every "agent" needs to be an LLM: those eight
lines are the whole supervisor.

In [ ]:
from agents import build_graph

graph = build_graph(llm, tools)
print(graph.get_graph().draw_mermaid())

## 4. Run the Client

One query as input, a figure as output. The task names the models and where to save — the *how* (which tools, in what order, passing file paths between steps) is up to the agents.

> A full run is ~10 model calls, but Gemini's free tier only allows a handful of requests per minute (and ~20 per day). 
When a 429 arrives, the client waits 60 s and retries (`_ainvoke` in `agents/nodes.py`) — so a run can take a few minutes. 

Paid or institutional endpoints don't need this.

In [ ]:
from agents import new_run

TASK = f"""Compute the linear matter power spectrum at z=0 
for three cosmologies: 
    (1) standard LCDM, 
    (2) LCDM with total neutrino mass 0.10 eV, and 
    (3) wCDM with w0=-0.9. 

Then plot all three against the eBOSS DR14 Lyman-alpha forest data,
using LCDM as the ratio reference. Save all files to {OUTPUT_DIR}."""

initial_state = new_run(TASK)

async for update in graph.astream(initial_state, stream_mode="updates"):
    for node, delta in update.items():
        print(f"=== {node} " + "=" * (60 - len(node)))
        if "plan" in delta:
            for step in delta["plan"]:
                print(f"  plan {step['id']}: {step['description']}")
        elif "final_report" in delta:
            final_report = delta["final_report"]
            print("  report ready")
        else:
            print(f"  {delta['step_results'][-1]}")

In [ ]:
from IPython.display import Markdown

Markdown(final_report)

## 5. Validation and verification 

The agent's figure next to the ground truth in Part 1. The
data points should sit *on* the LCDM curve. 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, (title, path) in zip(axes, [
    ("Agent output", f"{OUTPUT_DIR}/power_spectrum_comparison.png"),
    ("Ground truth", "../ground-truth/power_spectrum_comparison.png"),
]):
    ax.imshow(mpimg.imread(path))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6.Next steps:

You now have all components of a state-of-the-art agentic system: science functions → published as MCP tools → driven by a small planning/working reporting graph. 

To customize it, add your own module to the server's `tools/` package and this client discovers it. 

There are other functionalities in this repo that are not covered: 
1. the **stdio transport** (the client spawns the server as a subprocess — no terminal
needed)
2. **skills** (recipes the agents load on demand), **persistent
memory**
3. **follow-up queries**, and the **Groq backend**. 

Continue with
[`notebooks/03_next_steps.ipynb`](03_next_steps.ipynb) — it demos each one of these above. 